# Covasim — Esplorazione reti di contatto

Notebook per esplorare come Covasim costruisce e gestisce le reti di contatto (grafi) usate per simulare la diffusione del COVID-19.

**Contenuto:**
1. Installazione e import
2. Costruzione di una popolazione base
3. Esplorazione della struttura a layer
4. Conversione in grafo NetworkX
5. Visualizzazione del grafo
6. Aggiungere un layer personalizzato
7. Simulazione epidemia e plot risultati

## 1. Installazione e import

In [2]:
# Installa covasim se non presente
# !pip install covasim

import covasim as cv
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm

print(f'Covasim version: {cv.__version__}')

Covasim version: 3.1.7


## 2. Costruzione di una popolazione base

Covasim supporta tre tipi di popolazione:
- `random`: tutti i contatti sono casuali, senza struttura sociale
- `hybrid`: layer separati per famiglia, scuola, lavoro, comunità
- `synthpop`: popolazione sintetica realistica (richiede SynthPops)

In [ ]:
# Popolazione piccola per esplorazione rapida
POP_SIZE = 500

sim = cv.Sim(dict(
    pop_size=POP_SIZE,
    pop_type='hybrid',  
    # household + school + work + community
))
sim.initialize()

people = sim.people
print(f'Numero di agenti: {len(people)}')
print(f'Layer disponibili: {list(people.contacts.keys())}')

Initializing sim with 500 people for 60 days
Numero di agenti: 500
Layer disponibili: ['h', 's', 'w', 'c']


## 3. Esplorazione della struttura a layer

Ogni layer è una lista di archi `(p1[i], p2[i])` con peso `beta[i]` (probabilità di trasmissione).

In [6]:
layer_labels = {
    'h': 'Household (famiglia)',
    's': 'School (scuola)',
    'w': 'Work (lavoro)',
    'c': 'Community (comunità)'
}

for key, label in layer_labels.items():
    layer = people.contacts[key]
    n_edges = len(layer['p1'])
    avg_degree = (2 * n_edges) / POP_SIZE
    print(f"{label:30s} | archi: {n_edges:5d} | grado medio: {avg_degree:.2f}")

Household (famiglia)           | archi:   521 | grado medio: 2.08
School (scuola)                | archi:   932 | grado medio: 3.73
Work (lavoro)                  | archi:  2606 | grado medio: 10.42
Community (comunità)           | archi:  4945 | grado medio: 19.78


In [7]:
# Guarda i primi archi del layer household
h = people.contacts['h']
print('Primi 10 archi del layer Household:')
print(f"  p1 (sorgente): {h['p1'][:10]}")
print(f"  p2 (destinaz): {h['p2'][:10]}")
print(f"  beta (peso):   {h['beta'][:10]}")

Primi 10 archi del layer Household:
  p1 (sorgente): [ 0  4  6  8  8  9 11 11 12 14]
  p2 (destinaz): [ 1  5  7  9 10 10 12 13 13 16]
  beta (peso):   [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [ ]:
# Distribuzione del grado per ogni layer
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (key, label) in zip(axes, layer_labels.items()):
    layer = people.contacts[key]
    all_nodes = np.concatenate([layer['p1'], layer['p2']])
    degrees = np.bincount(all_nodes, minlength=POP_SIZE)
    ax.hist(degrees[degrees > 0], bins=20, color='steelblue', edgecolor='white')
    ax.set_title(label.split('(')[0].strip(), fontsize=11)
    ax.set_xlabel('Grado (n° contatti)')
    ax.set_ylabel('Frequenza')

plt.suptitle('Distribuzione del grado per layer', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Conversione in grafo NetworkX

Ogni layer può essere convertito in un `DiGraph`, e l'insieme dei layer in un `MultiDiGraph`.

In [ ]:
# Grafo del solo layer household
G_household = people.contacts['h'].to_graph()
print(f'Household — nodi: {G_household.number_of_nodes()}, archi: {G_household.number_of_edges()}')

# Grafo completo (tutti i layer)
G_all = people.contacts.to_graph()
print(f'Tutti i layer — nodi: {G_all.number_of_nodes()}, archi: {G_all.number_of_edges()}')

In [ ]:
# Statistiche del grafo household come grafo non diretto
G_h_undirected = G_household.to_undirected()

# Considera solo la componente connessa più grande
largest_cc = max(nx.connected_components(G_h_undirected), key=len)
G_h_cc = G_h_undirected.subgraph(largest_cc)

print(f'Componente connessa più grande: {len(largest_cc)} nodi')
print(f'Grado medio: {np.mean([d for _, d in G_h_cc.degree()]):.2f}')
print(f'Numero componenti connesse: {nx.number_connected_components(G_h_undirected)}')
print(f'Densità: {nx.density(G_h_undirected):.4f}')

## 5. Visualizzazione del grafo

Visualizziamo un sottoinsieme del grafo household, colorando i nodi per età.

In [ ]:
# Prendi i primi N nodi connessi per rendere la visualizzazione leggibile
N_VIZ = 80
nodes_to_viz = list(G_h_undirected.nodes())[:N_VIZ]
G_sub = G_h_undirected.subgraph(nodes_to_viz)

# Colora per età
ages = people.age[nodes_to_viz]
norm = plt.Normalize(vmin=ages.min(), vmax=ages.max())
colors = cm.plasma(norm(ages))

fig, ax = plt.subplots(figsize=(10, 8))
pos = nx.spring_layout(G_sub, seed=42)
nx.draw_networkx(
    G_sub, pos=pos,
    node_color=colors,
    node_size=120,
    edge_color='gray',
    alpha=0.8,
    with_labels=False,
    ax=ax
)
sm = plt.cm.ScalarMappable(cmap='plasma', norm=norm)
plt.colorbar(sm, ax=ax, label='Età')
ax.set_title(f'Layer Household — primi {N_VIZ} nodi (colorati per età)', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Aggiungere un layer personalizzato

Esempio: un layer `transport` che simula contatti sui mezzi pubblici.

In [ ]:
rng = np.random.default_rng(seed=0)

# Crea ~1 contatto casuale per persona sul trasporto
n_transport_contacts = POP_SIZE
p1 = rng.integers(0, POP_SIZE, size=n_transport_contacts)
p2 = rng.integers(0, POP_SIZE, size=n_transport_contacts)
# Rimuovi self-loops
mask = p1 != p2
p1, p2 = p1[mask], p2[mask]

# Nella versione 3.x il Layer è un dict-like: si popola con layer['key'] = ...
transport_layer = cv.Layer()
transport_layer['p1'] = p1
transport_layer['p2'] = p2
transport_layer['beta'] = np.full(len(p1), 0.3)  # trasmissione ridotta
transport_layer.label = 'transport'

# Aggiungi alla simulazione
sim.people.contacts.add_layer(transport=transport_layer)
print(f'Layer disponibili dopo aggiunta: {list(sim.people.contacts.keys())}')
print(f'Archi nel layer transport: {len(transport_layer["p1"])}')

## 7. Simulazione epidemia e plot risultati

In [ ]:
# Simulazione base senza interventi
sim_base = cv.Sim(dict(
    pop_size=POP_SIZE,
    pop_type='hybrid',
    n_days=120,
    beta=0.015,
    label='Nessun intervento'
))

# Simulazione con distanziamento sociale (riduzione contatti del 50%)
sim_dist = cv.Sim(dict(
    pop_size=POP_SIZE,
    pop_type='hybrid',
    n_days=120,
    beta=0.015,
    interventions=cv.change_beta(days=30, changes=0.5),
    label='Distanziamento dal giorno 30'
))

# Esegui entrambe
msim = cv.MultiSim([sim_base, sim_dist])
msim.run()
msim.plot()

In [ ]:
# Plot confronto infetti nel tempo
fig, ax = plt.subplots(figsize=(10, 5))

for s in msim.sims:
    ax.plot(s.results['t'], s.results['new_infections'], label=s.label)

ax.axvline(x=30, color='gray', linestyle='--', alpha=0.6, label='Inizio distanziamento')
ax.set_xlabel('Giorno')
ax.set_ylabel('Nuovi infetti')
ax.set_title('Confronto: nuovi infetti al giorno')
ax.legend()
plt.tight_layout()
plt.show()

## Note e prossimi passi

- Il grafo complessivo è la **sovrapposizione dei layer** — ogni layer cattura un contesto sociale diverso
- I pesi `beta` per layer si moltiplicano per il `beta` globale della simulazione
- Per collegare con EDRep: si può estrarre la matrice di adiacenza di un layer e passarla a `NodeEmbedding`

```python
# Esempio: embedding del grafo household con EDRep
import scipy.sparse as sp
from EDRep_main.EDRep import NodeEmbedding

h = sim.people.contacts['h']
A = sp.csr_matrix((np.ones(len(h['p1'])), (h['p1'], h['p2'])), shape=(POP_SIZE, POP_SIZE))
A = A + A.T  # rendi simmetrica
embedding = NodeEmbedding(A, dim=16)
```